# Exploratory Data Analysis & Machine Learning Benchmark (EDA)
## Cybersecurity – Detecting Phishing Emails

This notebook performs comprehensive Exploratory Data Analysis (EDA), hybrid TF-IDF + Metadata feature extraction, and a 4-model benchmark evaluation (`Logistic Regression`, `Naive Bayes`, `Random Forest`, `MLP Neural Network`) inspired by top Kaggle phishing detection pipelines.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


In [ ]:
df = pd.read_csv('data/data.csv')
print(f'Dataset Shape: {df.shape}')
print('Columns:', df.columns.tolist())
df.head()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df['phishing_email'].value_counts()
ax[0].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
ax[0].set_title('Target Class Ratios')

sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax[1], palette=['#2ecc71', '#e74c3c'])
ax[1].set_title('Target Class Distribution Count')
ax[1].set_ylabel('Number of Emails')

plt.tight_layout()
plt.show()


In [ ]:
def extract_metadata_features(text):
    text_str = str(text)
    url_count = len(re.findall(r'https?://\S+|www\.\S+', text_str))
    digit_count = sum(c.isdigit() for c in text_str)
    uppercase_count = sum(c.isupper() for c in text_str)
    special_count = len(re.findall(r'[^a-zA-Z0-9\s]', text_str))
    char_count = len(text_str)
    word_count = len(text_str.split())
    return [url_count, digit_count, uppercase_count, special_count, char_count, word_count]

metadata_list = [extract_metadata_features(t) for t in df['email_text']]
meta_cols = ['url_count', 'digit_count', 'uppercase_count', 'special_count', 'char_count', 'word_count']
meta_df = pd.DataFrame(metadata_list, columns=meta_cols)
meta_df['label'] = df['phishing_email'].map({'Legitimate': 0, 'Phishing': 1})

meta_df.head()


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(meta_df.corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Metadata Feature Correlation Heatmap')
plt.show()


In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words='english'
)

X_text = tfidf.fit_transform(df['email_text'].fillna(''))
X_meta = meta_df[meta_cols].values
X = hstack([X_text, X_meta])
y = meta_df['label'].values

print(f'Combined Feature Matrix Shape: {X.shape}')


In [ ]:
feature_names = tfidf.get_feature_names_out()
idf_scores = tfidf.idf_

tfidf_df = pd.DataFrame({'Word': feature_names, 'IDF': idf_scores})
top_words = tfidf_df.sort_values(by='IDF', ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.barh(top_words['Word'], top_words['IDF'], color='#3498db')
plt.title('Top 20 Important TF-IDF Words (by IDF Score)')
plt.xlabel('Inverse Document Frequency (IDF)')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training Samples : {X_train.shape[0]}')
print(f'Testing Samples  : {X_test.shape[0]}')


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
nb = MultinomialNB()
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=50, random_state=42)

models = {
    'Logistic Regression': lr,
    'Naive Bayes': nb,
    'Random Forest': rf,
    'Neural Network (MLP)': mlp
}

results = []
preds = {}
probs = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred
    
    preds[name] = y_pred
    probs[name] = y_prob
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
results_df


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for idx, (name, pred) in enumerate(preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
    axes[idx].set_title(name)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))

for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, prob):.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Model ROC Curves Benchmark')
plt.legend()
plt.show()


In [ ]:
def predict_email(email_text):
    text_vector = tfidf.transform([email_text])
    meta_vector = np.array([extract_metadata_features(email_text)])
    features = hstack([text_vector, meta_vector])
    
    prediction = lr.predict(features)[0]
    probability = lr.predict_proba(features)[0]
    
    confidence = max(probability) * 100
    label = 'Phishing' if prediction == 1 else 'Legitimate'
    
    print('=' * 55)
    print(f'Prediction : {label}')
    print(f'Confidence : {confidence:.2f}%')
    print('=' * 55)
    return prediction


In [ ]:
sample_legit = """
Hi Manish,

Tomorrow's project meeting will begin at 10 AM in Lab 302.

Please bring your presentation slides.

Thanks.

Professor
"""

predict_email(sample_legit)


In [ ]:
sample_phish = """
Dear Customer,

Your bank account has been suspended.

Please verify your account immediately by clicking the link below.

https://paypal-security-login.xyz

Failure to verify within 24 hours will permanently suspend your account.

Regards,
Security Team
"""

predict_email(sample_phish)
